In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("data/dataset.csv")

# EDA:

In [3]:
df['Disease_Prediction'].value_counts()

Disease_Prediction
Bovine Tuberculosis               58
Bovine Respiratory Disease        53
Swine Influenza                   45
Equine Influenza                  45
Caprine Arthritis Encephalitis    42
                                  ..
Chronic Bronchitis                 1
Canine Heartworm Disease           1
Canine Influenza                   1
Heartworm Disease                  1
Respiratory Syncytial Virus        1
Name: count, Length: 136, dtype: int64

In [4]:
print(df["Disease_Prediction"].value_counts()[df["Disease_Prediction"].value_counts() < 2])

Disease_Prediction
Inflammatory Bowel Disease     1
Goat Pox                       1
Cryptosporidiosis              1
Tuberculosis                   1
Chronic Bronchitis             1
Canine Heartworm Disease       1
Canine Influenza               1
Heartworm Disease              1
Respiratory Syncytial Virus    1
Name: count, dtype: int64


In [5]:
counts = df["Disease_Prediction"].value_counts()

df["Disease_Prediction"] = df["Disease_Prediction"].apply(
    lambda x: x if counts[x] > 2 else "Other"
)

In [6]:
from sklearn.model_selection import StratifiedShuffleSplit

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index, test_index in split.split(df, df["Disease_Prediction"]):
    strat_train_set = df.loc[train_index]
    strat_test_set = df.loc[test_index]

## Convert values like "102 F", "3 days" → 102 , 3

In [7]:
def extract_number(value):
    if pd.isna(value):
        return np.nan
    value = str(value)
    number = re.findall(r"\d+\.?\d*", value)

    if number:
        return float(number[0])
    else:
        return np.nan

## Convert different yes/no forms to standard values

In [8]:
def clean_yes_no(value):
    value = str(value).strip().lower()
    if value in ["yes", "y", "true", "1"]:
        return "Yes"
    if value in ["no", "n", "false", "0"]:
        return "No"
    return ""

## Standardize text (cow → Cow, german shepherd → German Shepherd)

In [9]:
def clean_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip().title()

# Main preprocessing function

In [10]:
def preprocess_df(df):
    df = df.copy()

    # Convert duration to number
    if "Duration" in df.columns:
        df["Duration_Days"] = df["Duration"].apply(extract_number)
        df.drop("Duration", axis=1, inplace=True)

    # Convert numeric text columns
    numeric_cols = ["Body_Temperature", "Heart_Rate"]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = df[col].apply(extract_number)

    # Convert Yes/No columns
    yes_no_cols = [
        "Appetite_Loss",
        "Vomiting",
        "Diarrhea",
        "Coughing",
        "Labored_Breathing",
        "Lameness",
        "Skin_Lesions",
        "Nasal_Discharge",
        "Eye_Discharge",
    ]

    for col in yes_no_cols:
        if col in df.columns:
            df[col] = df[col].apply(clean_yes_no)

    # Clean text columns
    text_cols = [
        "Animal_Type",
        "Breed",
        "Gender",
        "Symptom_1",
        "Symptom_2",
        "Symptom_3",
        "Symptom_4",
    ]

    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].apply(clean_text)

    return df

# Transforming data:

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report
import joblib
from preprocess_utils import preprocess_df

In [12]:
MODEL_PATH = "model.pkl"

In [13]:
df = pd.read_csv("data/dataset.csv")

In [14]:
import re

In [15]:
df = preprocess_df(df)

In [16]:
target = "Disease_Prediction"
if target not in df.columns:
    raise ValueError("Disease_Prediction column not found in dataset")

In [17]:
X = df.drop(columns=[target])         #Features
y = df[target]                        #Labels

In [18]:
numeric_features = [
        "Age",
        "Weight",
        "Duration_Days",
        "Body_Temperature",
        "Heart_Rate",
    ]

In [19]:
categorical_features = [c for c in X.columns if c not in numeric_features]

In [20]:
numeric_transformer = Pipeline(
    steps = [
            ("imputer", SimpleImputer(strategy="median"))
    ]
    )

In [21]:
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

In [22]:
preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )

# Using Decision Tree algo:

In [23]:
from sklearn.tree import DecisionTreeClassifier

In [24]:
model = DecisionTreeClassifier()

In [25]:
clf = Pipeline(
        steps=[
            ("prep", FunctionTransformer(preprocess_df, validate=False)),
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

In [26]:
class_counts = y.value_counts()
if class_counts.min() < 2:
        # Too few samples in some classes for stratified split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
else:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )

In [27]:
clf.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('prep', ...), ('preprocessor', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function pre...0012D1112CB80>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False
,"check_inverse check_inverse: bool, default=TrueWhether to check that or ``func`` followed by ``inverse_func`` leads tothe original inputs. It can be used for a sanity check, raising awarning when the condition is not fulfilled... versionadded:: 0.20",True
,"feature_names_out feature_names_out: callable, 'one-to-one' or None, default=NoneDetermines the list of feature names that will be returned by the`get_feature_names_out` method. If it is 'one-to-one', then the outputfeature names will be equal to the input feature names. If it is acallable, then it must take two positional arguments: this`FunctionTransformer` (`self`) and an array-like of input feature names(`input_features`). It must return an array-like of output featurenames. The `get_feature_names_out` method is only defined if`feature_names_out` is not None.See ``get_feature_names_out`` for more details... versionadded:: 1.1",None
,"kw_args kw_args: dict, default=NoneDictionary of additional keyword arguments to p

In [28]:
y_pred = clf.predict(X_test)

In [29]:
acc = accuracy_score(y_test, y_pred)

In [30]:
acc

0.86

In [31]:
print(f"Accuracy: {acc:.4f}")

Accuracy: 0.8600


In [32]:
# print(f"Classification Report :",classification_report(y_test, y_pred, zero_division=0))

# Using Random Forest:

In [33]:
from sklearn.ensemble import RandomForestClassifier

In [34]:
model = RandomForestClassifier()

In [35]:
clf = Pipeline(
        steps=[
            ("prep", FunctionTransformer(preprocess_df, validate=False)),
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

In [36]:
class_counts = y.value_counts()
if class_counts.min() < 2:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
else:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )

In [37]:
clf.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('prep', ...), ('preprocessor', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function pre...0012D1112CB80>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False
,"check_inverse check_inverse: bool, default=TrueWhether to check that or ``func`` followed by ``inverse_func`` leads tothe original inputs. It can be used for a sanity check, raising awarning when the condition is not fulfilled... versionadded:: 0.20",True
,"feature_names_out feature_names_out: callable, 'one-to-one' or None, default=NoneDetermines the list of feature names that will be returned by the`get_feature_names_out` method. If it is 'one-to-one', then the outputfeature names will be equal to the input feature names. If it is acallable, then it must take two positional arguments: this`FunctionTransformer` (`self`) and an array-like of input feature names(`input_features`). It must return an array-like of output featurenames. The `get_feature_names_out` method is only defined if`feature_names_out` is not None.See ``get_feature_names_out`` for more details... versionadded:: 1.1",None
,"kw_args kw_args: dict, default=NoneDictionary of additional keyword arguments to p

In [38]:
y_pred = clf.predict(X_test)

In [39]:
acc = accuracy_score(y_test, y_pred)

In [40]:
acc

0.9

In [41]:
print(f"Accuracy: {acc:.4f}")

Accuracy: 0.9000


In [42]:
print(f"Accuracy: {acc:.4f}")

Accuracy: 0.9000


In [43]:
# Classification Report:

In [44]:
# print(classification_report(y_test, y_pred, zero_division=0))

## Using Logistic Regression

In [45]:
df.head(3)

,Animal_Type,Breed,Age,Gender,Weight,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Appetite_Loss,...,Coughing,Labored_Breathing,Lameness,Skin_Lesions,Nasal_Discharge,Eye_Discharge,Body_Temperature,Heart_Rate,Disease_Prediction,Duration_Days
0,Cat,Persian,7,Female,4,Sneezing,Nasal Discharge,Eye Discharge,Lethargy,Yes,...,Yes,Yes,No,Yes,No,Yes,38.9,141.999125,Feline Upper Respiratory Infection,4.0
1,Dog,Shih Tzu,4,Male,8,Lethargy,Vomiting,Loss Of Appetite,Fever,Yes,...,Yes,No,No,Yes,No,Yes,39.2,115.014160,Parvovirus,4.0
2,Dog,Chihuahua,4,Female,5,Lethargy,Vomiting,Loss Of Appetite,Fever,Yes,...,Yes,Yes,Yes,Yes,No,No,39.1,110.001877,Canine Parvovirus,3.0


In [46]:
from sklearn.preprocessing import LabelEncoder

In [47]:
X = df.drop("Disease_Prediction", axis=1)
y = df["Disease_Prediction"]

In [48]:
# le_animal = LabelEncoder()
# df['Animal_Type'] = le_animal.fit_transform(df['Animal_Type'])

# animal_type_encoding = dict(zip(le_animal.classes_, le_animal.transform(le_animal.classes_)))
# print("Animal_Type Encoding:", animal_type_encoding)

In [49]:
X = pd.get_dummies(X)

In [50]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [51]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [52]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

lr = LogisticRegression(max_iter=1000)

lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

acc_lr = accuracy_score(y_test, y_pred_lr)

print("Logistic Regression Accuracy:", acc_lr)

Logistic Regression Accuracy: 0.8933333333333333


# Using SVM:

In [53]:
from sklearn.svm import SVC

svm = SVC(kernel='rbf')  # you can try 'linear' also

svm.fit(X_train, y_train)

y_pred_svm = svm.predict(X_test)

acc_svm = accuracy_score(y_test, y_pred_svm)

print("SVM Accuracy:", acc_svm)

SVM Accuracy: 0.77


## Difference in the o/p

In [55]:
import pandas as pd

In [56]:
df = pd.read_csv("data/output.csv")

In [57]:
df.head()

,Expected,Predicted
0,Caprine Arthritis Encephalitis,Caprine Arthritis Encephalitis
1,Porcine Epidemic Diarrhea Virus,Porcine Epidemic Diarrhea Virus
2,Bovine Respiratory Disease,Bovine Respiratory Disease
3,Canine Leptospirosis,Canine Leptospirosis
4,Canine Hepatitis,Canine Hepatitis


In [58]:
error = df["Expected"] != df["Predicted"]

In [59]:
df[df["Expected"] != df["Predicted"]]

,Expected,Predicted
39,Swine Fever,Swine Influenza
64,Tuberculosis,Bovine Respiratory Disease
84,Swine Dysentery,Swine Influenza
101,Feline Leukemia,Conjunctivitis
106,Feline Herpesvirus,Feline Calicivirus
107,Bovine Tuberculosis,Bovine Respiratory Disease
118,Bovine Tuberculosis,Bovine Respiratory Disease
135,Equine Herpesvirus,Equine Leptospirosis
192,Kennel Cough,Canine Cough
195,Equine Influenza,West Nile Virus
